<a href="https://colab.research.google.com/github/nataliamarinn/labo3-2026r/blob/main/src/AutoGluon/z335_DistribucionFit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Predicción por ajuste de distribución

## La idea

En vez de modelar la serie como proceso temporal, tratamos los valores históricos como **muestras de una distribución** y predecimos el valor más probable (moda) o el valor central (mediana).

Esto ignora el orden temporal — pero si la serie es mayormente ruido alrededor de un nivel estable, el valor más frecuente es una predicción tan buena como cualquier modelo de tendencia.

```
serie: [12, 15, 11, 98, 13, 14, 12]  ← el 98 es outlier

OLS / HAR:   anchado al nivel alto por el outlier
KDE mode:    pico en ~13  (ignora el outlier, busca el valor más frecuente)
```

## Modelos

| Modelo | Predicción | Ventaja |
|---|---|---|
| **LogNormal** | moda = exp(μ − σ²) | Natural para ventas (positivo, asimétrico) |
| **Gamma** | moda = (α−1)/β | Similar pero cola menos pesada |
| **KDE mode** | pico de la densidad empírica | Sin supuesto de forma, detecta bimodalidad |
| **KDE mode reciente** | pico sobre últimos N meses | Como KDE pero anclado al nivel nuevo |

## Backtesting
Train hasta 201910, target 201912.

## 0. Init Google Colab

In [ ]:
from google.colab import drive
drive.mount('/content/.drive')

In [ ]:
%%shell

mkdir -p "/content/.drive/My Drive/labo3"
mkdir -p "/content/buckets"
ln -sfn "/content/.drive/My Drive/labo3"   /content/buckets/b1

mkdir -p ~/.kaggle
cp /content/buckets/b1/kaggle/kaggle.json  ~/.kaggle
chmod 600 ~/.kaggle/kaggle.json

mkdir -p /content/buckets/b1/exp
mkdir -p /content/buckets/b1/datasets
mkdir -p /content/datasets

descargar() {
  carpeta_destino="/content/buckets/b1/datasets/"
  url_origen="https://storage.googleapis.com/open-courses/austral2026-5da5/labo3/"
  archivo="$1"

  if ! test -f "$carpeta_destino""$archivo"; then
    wget  "$url_origen""$archivo"  -O "$carpeta_destino""$archivo"
  fi

  if ! test -f  "/content/datasets/""$archivo"; then
    cp  "$carpeta_destino""$archivo"  "/content/datasets/""$archivo"
  fi;
}

descargar  "sell-in.txt.gz"
descargar  "product_id_apredecir201912.txt"

# 1. Setup

In [ ]:
!pip install uv
!uv pip install -q kaggle

In [ ]:
import os
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import chi2
from sklearn.neighbors import KernelDensity

import warnings
warnings.filterwarnings('ignore')

In [ ]:
PARAM = {
    'experimento':        'DistribucionFit-01',
    'kaggle_competition': 'labo-iii-2026-rosario',
    'ventana_kde':        12,   # meses para KDE reciente
    'kde_bandwidth':      'scott',  # 'scott', 'silverman', o float
    'periodo_corte':      201910,
    'periodo_target':     201912,
}

ruta = "/content/buckets/b1/exp/" + PARAM['experimento']
os.makedirs(ruta, exist_ok=True)
os.chdir(ruta)

# 2. Datos

In [ ]:
dataset = pl.read_csv('/content/.drive/My Drive/labo3/datasets/sell-in.txt.gz', separator="\t")

tb_ventas = dataset.group_by("product_id", "periodo").agg(
    pl.col("tn").sum().alias("tn")
).sort(["product_id", "periodo"])

tb_apredecir = pl.read_csv('/content/.drive/My Drive/labo3/datasets/product_id_apredecir201912.txt', separator="\t")
tb_ventas    = tb_ventas.join(tb_apredecir, on="product_id", how="inner").sort(["product_id", "periodo"])

tb_train = tb_ventas.filter(pl.col("periodo") <= PARAM['periodo_corte'])
tb_real  = (
    tb_ventas
    .filter(pl.col("periodo") == PARAM['periodo_target'])
    .select(["product_id", "tn"])
    .rename({"tn": "tn_real"})
)

productos = tb_apredecir["product_id"].to_list()
print(f"{len(productos)} productos")

# 3. Funciones de predicción por distribución

Cada función toma la serie histórica y devuelve un escalar: el valor más probable según la distribución ajustada.

In [ ]:
def pred_lognormal_mode(serie: np.ndarray) -> float:
    """
    Fitea LogNormal a la serie (tratando los valores como muestras iid).
    Predicción = moda = exp(μ - σ²)
    Usa log(tn + 1) para manejar ceros.
    """
    s = serie[serie > 0]
    if len(s) < 3:
        return max(float(serie.mean()), 0.0)
    log_s = np.log(s)
    mu    = log_s.mean()
    sigma2 = log_s.var()
    moda  = np.exp(mu - sigma2)
    return max(float(moda), 0.0)


def pred_lognormal_median(serie: np.ndarray) -> float:
    """
    LogNormal mediana = exp(μ)  — más estable que la moda cuando σ es grande.
    """
    s = serie[serie > 0]
    if len(s) < 3:
        return max(float(serie.mean()), 0.0)
    mu = np.log(s).mean()
    return max(float(np.exp(mu)), 0.0)


def pred_gamma_mode(serie: np.ndarray) -> float:
    """
    Fitea Gamma (MLE) y predice la moda = (α-1)/β.
    Si α <= 1 (distribución en J inversa), cae a la mediana empírica.
    """
    s = serie[serie > 0]
    if len(s) < 3:
        return max(float(serie.mean()), 0.0)
    try:
        alpha, loc, beta = stats.gamma.fit(s, floc=0)
        if alpha <= 1:
            return max(float(np.median(s)), 0.0)
        moda = (alpha - 1) * beta
        return max(float(moda), 0.0)
    except Exception:
        return max(float(np.median(s)), 0.0)


def pred_kde_mode(serie: np.ndarray, bandwidth='scott') -> float:
    """
    Ajusta KDE sobre todos los valores históricos y devuelve el pico (moda).
    No asume forma paramétrica — detecta bimodalidad.
    """
    s = serie[serie > 0]
    if len(s) < 3:
        return max(float(serie.mean()), 0.0)

    if bandwidth == 'scott':
        bw = 1.06 * s.std() * len(s) ** (-1/5)
    elif bandwidth == 'silverman':
        bw = 0.9 * min(s.std(), (np.percentile(s,75)-np.percentile(s,25))/1.34) * len(s)**(-1/5)
    else:
        bw = float(bandwidth)

    bw = max(bw, 1e-3)
    kde = KernelDensity(bandwidth=bw, kernel='gaussian')
    kde.fit(s.reshape(-1, 1))

    grid = np.linspace(s.min(), s.max(), 500).reshape(-1, 1)
    log_dens = kde.score_samples(grid)
    moda = float(grid[np.argmax(log_dens)][0])
    return max(moda, 0.0)


def pred_kde_mode_reciente(serie: np.ndarray, ventana: int,
                            bandwidth='scott') -> float:
    """
    Igual que KDE mode pero solo sobre los últimos `ventana` meses.
    Si hubo un cambio de nivel, el KDE reciente va a tener su pico en el nuevo nivel.
    """
    w = min(ventana, len(serie))
    return pred_kde_mode(serie[-w:], bandwidth=bandwidth)


print("Funciones definidas")

# 4. Visualización — ¿qué hace cada distribución?

Antes de correr el backtesting, miramos 6 productos al azar para entender visualmente cómo se comporta cada modelo.

In [ ]:
np.random.seed(42)
muestra_pids = np.random.choice(productos, 6, replace=False).tolist()

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, pid in enumerate(muestra_pids):
    serie = (
        tb_train.filter(pl.col("product_id") == pid)
        .sort("periodo")["tn"].to_numpy().astype(float)
    )

    s = serie[serie > 0]
    ax = axes[i]

    # histograma de los valores
    ax.hist(s, bins=20, density=True, color='lightsteelblue',
            edgecolor='white', alpha=0.8, label='distribución empírica')

    x_range = np.linspace(s.min() * 0.5, s.max() * 1.2, 300)

    # LogNormal
    mu_ln = np.log(s).mean()
    sg_ln = np.log(s).std()
    if sg_ln > 0:
        ax.plot(x_range, stats.lognorm.pdf(x_range, s=sg_ln, scale=np.exp(mu_ln)),
                color='tomato', linewidth=1.5, label='LogNormal')

    # Gamma
    try:
        alpha, loc, beta = stats.gamma.fit(s, floc=0)
        ax.plot(x_range, stats.gamma.pdf(x_range, alpha, loc=0, scale=beta),
                color='green', linewidth=1.5, label='Gamma')
    except Exception:
        pass

    # KDE
    bw = 1.06 * s.std() * len(s)**(-1/5)
    bw = max(bw, 1e-3)
    kde = KernelDensity(bandwidth=bw, kernel='gaussian').fit(s.reshape(-1,1))
    log_d = kde.score_samples(x_range.reshape(-1,1))
    ax.plot(x_range, np.exp(log_d), color='purple', linewidth=1.5,
            linestyle='--', label='KDE')

    # predicciones verticales
    for val, color, label in [
        (pred_lognormal_mode(serie),      'tomato',  'LN moda'),
        (pred_gamma_mode(serie),          'green',   'Gamma moda'),
        (pred_kde_mode(serie),            'purple',  'KDE moda'),
        (pred_kde_mode_reciente(serie,12),'orange',  'KDE 12m'),
        (float(np.median(serie)),         'gray',    'mediana'),
    ]:
        ax.axvline(val, color=color, linestyle=':', linewidth=1.2)

    ax.set_title(f'pid {pid}  (n={len(s)})', fontsize=8)
    ax.legend(fontsize=5)

fig.suptitle('Distribuciones ajustadas por producto — líneas punteadas = predicciones', fontsize=10)
plt.tight_layout()
plt.show()

# 5. Backtesting — todos los modelos

In [ ]:
bw  = PARAM['kde_bandwidth']
vkde = PARAM['ventana_kde']

resultados = []

for pid in productos:
    serie = (
        tb_train.filter(pl.col("product_id") == pid)
        .sort("periodo")["tn"].to_numpy().astype(float)
    )

    resultados.append({
        'product_id':       pid,
        'pred_naive':       max(float(np.median(serie[-6:])), 0.0),
        'pred_media3m':     max(float(serie[-3:].mean()),     0.0),
        'pred_ln_moda':     pred_lognormal_mode(serie),
        'pred_ln_mediana':  pred_lognormal_median(serie),
        'pred_gamma_moda':  pred_gamma_mode(serie),
        'pred_kde_full':    pred_kde_mode(serie, bandwidth=bw),
        'pred_kde_rec':     pred_kde_mode_reciente(serie, vkde, bandwidth=bw),
        'pred_kde_rec3':    pred_kde_mode_reciente(serie, 3,    bandwidth=bw),
    })

tb_preds = pl.DataFrame(resultados)
print("Predicciones listas")

In [ ]:
tb_bt = tb_real.join(tb_preds, on='product_id', how='left')

modelos = ['naive', 'media3m', 'ln_moda', 'ln_mediana', 'gamma_moda',
           'kde_full', 'kde_rec', 'kde_rec3']

for m in modelos:
    tb_bt = tb_bt.with_columns(
        (pl.col('tn_real') - pl.col(f'pred_{m}')).abs().alias(f'err_{m}')
    )

print("RMSE — backtesting 201912")
print()
rmse_naive = float(np.sqrt((tb_bt['err_naive'] ** 2).mean()))
rmse_dict  = {}
for m in modelos:
    rmse = float(np.sqrt((tb_bt[f'err_{m}'] ** 2).mean()))
    rmse_dict[m] = rmse
    delta = rmse - rmse_naive
    tag = '(baseline)' if m == 'naive' else f'({delta:+.4f} vs naive)'
    print(f"  {m:15s}: {rmse:.4f}  {tag}")

# 6. ¿Cuándo gana KDE vs LogNormal?

KDE debería ganar en series bimodales (quiebre de nivel) porque detecta el pico nuevo.
LogNormal debería ganar en series limpias con distribución unimodal.

Medimos la bimodalidad con el **coeficiente de bimodalidad** de Pfister et al.:
`BC = (skewness² + 1) / (kurtosis + 3(n-1)² / ((n-2)(n-3)))`
BC > 0.555 → distribución bimodal.

In [ ]:
from scipy.stats import skew, kurtosis

bimod = []
for pid in productos:
    serie = (
        tb_train.filter(pl.col("product_id") == pid)
        .sort("periodo")["tn"].to_numpy().astype(float)
    )
    s = serie[serie > 0]
    n = len(s)
    if n < 4:
        bc = 0.0
    else:
        sk = skew(s)
        ku = kurtosis(s)  # excess kurtosis
        correction = 3 * (n-1)**2 / ((n-2)*(n-3))
        bc = (sk**2 + 1) / (ku + correction)
    bimod.append({'product_id': pid, 'bimod_coef': float(bc)})

tb_bt = tb_bt.join(pl.DataFrame(bimod), on='product_id', how='left')

# split: bimodales vs unimodales
UMBRAL_BC = 0.555
tb_bim = tb_bt.filter(pl.col('bimod_coef') >  UMBRAL_BC)
tb_uni = tb_bt.filter(pl.col('bimod_coef') <= UMBRAL_BC)

print(f"Bimodales  (BC > {UMBRAL_BC}): {tb_bim.height} productos")
print(f"Unimodales (BC ≤ {UMBRAL_BC}): {tb_uni.height} productos")
print()

for nombre, grupo in [('BIMODALES', tb_bim), ('UNIMODALES', tb_uni)]:
    print(f"--- {nombre} ---")
    for m in ['naive', 'media3m', 'ln_moda', 'gamma_moda', 'kde_full', 'kde_rec', 'kde_rec3']:
        rmse = float(np.sqrt((grupo[f'err_{m}'] ** 2).mean()))
        print(f"  {m:15s}: {rmse:.4f}")
    print()

# 7. Visualización de la densidad en productos bimodales

Los productos con BC > 0.555 son los más interesantes: el KDE debería mostrar dos picos.

In [ ]:
top_bimod = (
    tb_bt.sort('bimod_coef', descending=True)
    .head(6)['product_id'].to_list()
)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, pid in enumerate(top_bimod):
    serie_full = tb_ventas.filter(pl.col('product_id') == pid).sort('periodo')
    periodos_  = serie_full['periodo'].to_list()
    tn_        = serie_full['tn'].to_numpy().astype(float)

    idx_corte  = next((j for j, p in enumerate(periodos_) if p > PARAM['periodo_corte']), len(periodos_))
    tn_train_  = tn_[:idx_corte]
    s          = tn_train_[tn_train_ > 0]

    row      = tb_bt.filter(pl.col('product_id') == pid)
    real_val = float(row['tn_real'][0])
    bc_val   = float(row['bimod_coef'][0])

    ax = axes[i]

    # densidad KDE
    bw_ = 1.06 * s.std() * len(s)**(-1/5)
    bw_ = max(bw_, 1e-3)
    kde_ = KernelDensity(bandwidth=bw_, kernel='gaussian').fit(s.reshape(-1,1))
    x_range = np.linspace(s.min()*0.3, s.max()*1.2, 500)
    log_d   = kde_.score_samples(x_range.reshape(-1,1))

    ax.fill_between(x_range, np.exp(log_d), alpha=0.3, color='purple')
    ax.plot(x_range, np.exp(log_d), color='purple', linewidth=1.5, label='KDE')

    for val, color, label in [
        (real_val,                          'black',  f'real={real_val:.1f}'),
        (pred_kde_mode(tn_train_),          'purple', f'KDE={pred_kde_mode(tn_train_):.1f}'),
        (pred_kde_mode_reciente(tn_train_,12),'orange',f'KDE12m={pred_kde_mode_reciente(tn_train_,12):.1f}'),
        (pred_lognormal_mode(tn_train_),    'tomato', f'LN={pred_lognormal_mode(tn_train_):.1f}'),
        (pred_gamma_mode(tn_train_),        'green',  f'Gamma={pred_gamma_mode(tn_train_):.1f}'),
    ]:
        ax.axvline(val, color=color, linewidth=1.5,
                   linestyle='-' if color == 'black' else '--')
        ax.text(val, ax.get_ylim()[1]*0.9 if i==0 else ax.get_ylim()[1]*0.01,
                f' {label}', color=color, fontsize=6, rotation=90, va='bottom')

    ax.set_title(f'pid {pid}  BC={bc_val:.3f}', fontsize=8)
    ax.set_xlabel('tn', fontsize=7)

fig.suptitle('Productos bimodales — dos picos en la densidad = quiebre de nivel', fontsize=10)
plt.tight_layout()
plt.show()

# 8. McNemar — significancia estadística

In [ ]:
def mcnemar(err_a, err_b, nombre_a, nombre_b):
    n10 = (err_a < err_b).sum()
    n01 = (err_b < err_a).sum()
    if n10 + n01 == 0:
        print(f"{nombre_a} vs {nombre_b}: sin discrepancias")
        return
    chi2_stat = (abs(n10 - n01) - 1)**2 / (n10 + n01)
    pvalue    = 1 - chi2.cdf(chi2_stat, df=1)
    ganador   = nombre_a if n10 > n01 else nombre_b
    sig = "** SIG **" if pvalue < 0.05 else "no sig   "
    print(f"{nombre_a:15s} vs {nombre_b:15s}:  gana {n10:3d}|{n01:3d}  p={pvalue:.4f}  {sig}  → {ganador}")

print("McNemar global (780 productos):")
print()
err_naive = tb_bt['err_naive'].to_numpy()
for m in ['media3m', 'ln_moda', 'ln_mediana', 'gamma_moda', 'kde_full', 'kde_rec', 'kde_rec3']:
    mcnemar(err_naive, tb_bt[f'err_{m}'].to_numpy(), 'naive', m)

print()
print("McNemar — mejor modelo vs resto:")
mejor_m = min(rmse_dict, key=rmse_dict.get)
print(f"  (mejor modelo en RMSE: {mejor_m} = {rmse_dict[mejor_m]:.4f})")
print()
err_mejor = tb_bt[f'err_{mejor_m}'].to_numpy()
for m in modelos:
    if m != mejor_m:
        mcnemar(err_mejor, tb_bt[f'err_{m}'].to_numpy(), mejor_m, m)

# 9. Modelo adaptativo: distribución según bimodalidad

Si BC > 0.555 (bimodal) → usar KDE reciente (el pico del nuevo nivel)

Si BC ≤ 0.555 (unimodal) → usar LogNormal mode (distribución limpia)

In [ ]:
tb_bt_dict = {row['product_id']: row for row in tb_bt.to_dicts()}

preds_adapt = []
for pid in productos:
    row = tb_bt_dict[pid]
    bc  = row['bimod_coef']
    pred = row['pred_kde_rec'] if bc > UMBRAL_BC else row['pred_ln_moda']
    preds_adapt.append({'product_id': pid, 'pred_adaptativo': pred})

tb_bt = tb_bt.join(pl.DataFrame(preds_adapt), on='product_id', how='left')
tb_bt = tb_bt.with_columns(
    (pl.col('tn_real') - pl.col('pred_adaptativo')).abs().alias('err_adaptativo')
)

rmse_adapt = float(np.sqrt((tb_bt['err_adaptativo'] ** 2).mean()))
print(f"RMSE adaptativo (KDE si bimodal, LN si unimodal): {rmse_adapt:.4f}")
print(f"RMSE naive:   {rmse_naive:.4f}")
print(f"RMSE media3m: {rmse_dict['media3m']:.4f}")

# 10. Submit

In [ ]:
# Cambiá modelo_submit para probar cada variante
PARAM_SUBMIT = {
    'modelo_submit': 'kde_rec',  # 'ln_moda', 'ln_mediana', 'gamma_moda',
                                  # 'kde_full', 'kde_rec', 'kde_rec3', 'adaptativo'
    'ventana_kde': PARAM['ventana_kde'],
}

tb_full     = tb_ventas
preds_final = []

for pid in productos:
    serie = (
        tb_full.filter(pl.col("product_id") == pid)
        .sort("periodo")["tn"].to_numpy().astype(float)
    )

    modelo = PARAM_SUBMIT['modelo_submit']

    if modelo == 'ln_moda':
        pred = pred_lognormal_mode(serie)
    elif modelo == 'ln_mediana':
        pred = pred_lognormal_median(serie)
    elif modelo == 'gamma_moda':
        pred = pred_gamma_mode(serie)
    elif modelo == 'kde_full':
        pred = pred_kde_mode(serie)
    elif modelo == 'kde_rec':
        pred = pred_kde_mode_reciente(serie, PARAM_SUBMIT['ventana_kde'])
    elif modelo == 'kde_rec3':
        pred = pred_kde_mode_reciente(serie, 3)
    elif modelo == 'adaptativo':
        s = serie[serie > 0]
        n = len(s)
        if n >= 4:
            sk = skew(s); ku = kurtosis(s)
            bc = (sk**2 + 1) / (ku + 3*(n-1)**2/((n-2)*(n-3)))
        else:
            bc = 0.0
        pred = pred_kde_mode_reciente(serie, PARAM_SUBMIT['ventana_kde']) if bc > UMBRAL_BC \
               else pred_lognormal_mode(serie)
    else:
        pred = max(float(np.median(serie[-6:])), 0.0)

    preds_final.append({'product_id': pid, 'tn': pred})

tb_final = pl.DataFrame(preds_final)
display(tb_final.head(10))
print(f"Nulls: {tb_final['tn'].is_null().sum()}")

In [ ]:
def kaggle_submit(competencia, archivo, mensaje):
    os.system(f'kaggle competitions submit -c {competencia} -f {archivo} -m "{mensaje}"')

modelo  = PARAM_SUBMIT['modelo_submit']
archivo = f"Distrib_{modelo}.csv"
mensaje = f"Distribucion {modelo} → moda/mediana distribución ajustada"

tb_final.write_csv(archivo)
kaggle_submit(PARAM['kaggle_competition'], archivo, mensaje)